In [1]:
%reset -f

import numpy as np
pi = np.pi
deg = np.degrees
sin = np.sin
cos = np.cos
tan = np.tan

arcsin = np.arcsin
arccos = np.arccos
arctan = np.arctan

%run "1维杆函数定义.ipynb"
%run "2维pin_joint_函数定义.ipynb"
%run "2维Plane Beam函数定义.ipynb"



# 1维杆求解 (单向拉压 u)

In [44]:
'''变量定义'''
EA = 1000.0
L = 1.0
P = 100.0

# 单元刚度矩阵
k1 = stiffness_1d(2 * EA, L)
k2 = stiffness_1d(EA, L)

print(f'k1 = \n{np.round(k1, 3)}')
print(f'k2 = \n{np.round(k2, 3)}')

# 总体刚度矩阵
K = assemble_K_1d_chain([k1, k2])

print(f'K = \n{np.round(K, 9)}')


# 位移边界条件
disp_bc = {
    1: 0.0,   # u1 = 0
    3: 0.0    # u3 = 0
}

# 力边界条件：直接写节点编号
force_bc = {
    2: P      # F2 = P
}

d, reaction, free_dofs, fixed_dofs = solve_structure_1d(
    K,
    force_bc=force_bc,
    disp_bc=disp_bc,
    numbering="one"
)

print("\nd =")
print(np.round(d, 9))

print("\nreaction =")
print(np.round(reaction, 9))

k1 = 
[[ 2000. -2000.]
 [-2000.  2000.]]
k2 = 
[[ 1000. -1000.]
 [-1000.  1000.]]
K = 
[[ 2000. -2000.     0.]
 [-2000.  3000. -1000.]
 [    0. -1000.  1000.]]

d =
[0.         0.03333333 0.        ]

reaction =
[-66.66666667   0.         -33.33333333]


# 2维pin_joint求解 (桁架, 两个方向的位移 u, v)

In [8]:
'''单元刚度'''
# k = stiffness(EA, L, theta)
k1 = stiffness(800*2.5e4, 4, -53.13, degree=True)
k2 = stiffness(480*2.5e4, 2.4, 0, degree=True)
k3 = stiffness(600*2.5e4, 3, 36.8699, degree=True)

print(f'k1 = \n{np.round(k1, 9)}')
print(f'k2 = \n{np.round(k2, 9)}')
print(f'k3 = \n{np.round(k3, 9)}')

'''组装总体刚度'''
# 单元的全局刚度矩阵列表
ke_list = [k1, k2, k3]

# 每个单元连接的节点
connectivity = [(1, 4), (2, 4), (3, 4)]

# 总体全局刚度矩阵
K = assemble_K(ke_list, connectivity, n_nodes=4, numbering="one")

print(f'K = \n{np.round(K, 9)}')

'''边界条件'''
# 位移边界条件
disp_bc = {
    dof(1, "u"): 0.0,
    dof(1, "v"): 0.0,
    dof(2, "u"): 0.0,
    dof(2, "v"): 0.0,
    dof(3, "u"): 0.0,
    dof(3, "v"): 0.0
}

# 受力边界条件
force_bc = {
    dof(4, "u"): 40e3,

}

'''求解'''
d, reaction, free_dofs, fixed_dofs = solve_structure(
    K,
    force_bc=force_bc,
    disp_bc=disp_bc
)

print("d =")
print(np.round(d, 9))

print("reaction =")
print(np.round(reaction, 9))


k1 = 
[[ 1800008.57480619 -2400002.50096852 -1800008.57480619  2400002.50096852]
 [-2400002.50096852  3199991.42519381  2400002.50096852 -3199991.42519381]
 [-1800008.57480619  2400002.50096852  1800008.57480619 -2400002.50096852]
 [ 2400002.50096852 -3199991.42519381 -2400002.50096852  3199991.42519381]]
k2 = 
[[ 5000000.        0. -5000000.       -0.]
 [       0.        0.       -0.       -0.]
 [-5000000.       -0.  5000000.        0.]
 [      -0.       -0.        0.        0.]]
k3 = 
[[ 3199999.80277869  2400000.05752287 -3199999.80277869 -2400000.05752287]
 [ 2400000.05752287  1800000.19722131 -2400000.05752287 -1800000.19722131]
 [-3199999.80277869 -2400000.05752287  3199999.80277869  2400000.05752287]
 [-2400000.05752287 -1800000.19722131  2400000.05752287  1800000.19722131]]
K = 
[[ 1.80000857e+06 -2.40000250e+06  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00 -1.80000857e+06  2.40000250e+06]
 [-2.40000250e+06  3.19999143e+06  0.00000000e+00  0.00000000e+00
   

In [4]:
'''杆状态'''
EA_list = [
    800*2.5e4,
    480*2.5e4,
    600*2.5e4
]

L_list = [4, 2.4, 3]
theta_list = [-53.13, 0, 36.8699]

results = bar_results_2d(
    d,
    connectivity,
    EA_list=EA_list,
    L_list=L_list,
    theta_list=theta_list,
    A_list=[800e-6, 480e-6, 600e-6],   # 如果不需要应力，可以删掉这一行
    degree=True,
    numbering="one"
)

for r in results:
    stress_text = "None" if r["stress"] is None else f"{r['stress']:.9e}"

    print(
        f"杆 {r['element']} ({r['node_i']}-{r['node_j']}): "
        f"伸缩量 = {r['elongation']:.9e}, "
        f"应变 = {r['strain']:.9e}, "
        f"应力 = {stress_text}, "
        f"轴力 = {r['force']:.9e}, "
        f"{r['state']}"
    )

杆 1 (1-4): 伸缩量 = 2.400002142e-03, 应变 = 6.000005355e-04, 应力 = 1.500001339e+07, 轴力 = 1.200001071e+04, 受拉
杆 2 (2-4): 伸缩量 = 3.999996649e-03, 应变 = 1.666665270e-03, 应力 = 4.166663176e+07, 轴力 = 1.999998324e+04, 受拉
杆 3 (3-4): 伸缩量 = 3.199998393e-03, 应变 = 1.066666131e-03, 应力 = 2.666665328e+07, 轴力 = 1.599999197e+04, 受拉


# 2维 Plane Beam (弯曲和横向位移 θ，v)

In [49]:
'''变量定义'''
#k = stiffness_beam(EI, L)

# 单元全局刚度矩阵
k1 = stiffness_beam(162000, 3)
k2 = stiffness_beam(162000, 3)

print(f'k1 = \n{np.round(k1, 9)}')
print(f'k2 = \n{np.round(k2, 9)}')

'''组装总体刚度'''
# 单元的全局刚度矩阵列表
ke_list = [k1, k2]

# 每个单元连接的节点
connectivity = [(1, 2), (2, 3)]

# 总体全局刚度矩阵
K = assemble_K(ke_list, connectivity, n_nodes=3, numbering="one")

print(f'K = \n{np.round(K, 9)}')

'''边界条件'''
disp_bc = {
    dof_beam(1, "v"): 0.0,
    dof_beam(1, "theta"): 0.0,
    
    dof_beam(2, "v"): 0.0,

    dof_beam(3, "v"): 0.0,
    dof_beam(3, "theta"): 0.0
}

force_bc = {
    dof_beam(1, "v"): -33.79,
    dof_beam(1, "theta"): -16.89,
    dof_beam(2, "v"): -33.79,
    dof_beam(2, "theta"): 16.89
}

'''求解'''
d, reaction, free_dofs, fixed_dofs = solve_structure(
    K,
    force_bc=force_bc,
    disp_bc=disp_bc
)

print("d =")
print(np.round(d, 9))

print("reaction =")
print(np.round(reaction, 9))


k1 = 
[[  72000.  108000.  -72000.  108000.]
 [ 108000.  216000. -108000.  108000.]
 [ -72000. -108000.   72000. -108000.]
 [ 108000.  108000. -108000.  216000.]]
k2 = 
[[  72000.  108000.  -72000.  108000.]
 [ 108000.  216000. -108000.  108000.]
 [ -72000. -108000.   72000. -108000.]
 [ 108000.  108000. -108000.  216000.]]
K = 
[[  72000.  108000.  -72000.  108000.       0.       0.]
 [ 108000.  216000. -108000.  108000.       0.       0.]
 [ -72000. -108000.  144000.       0.  -72000.  108000.]
 [ 108000.  108000.       0.  432000. -108000.  108000.]
 [      0.       0.  -72000. -108000.   72000. -108000.]
 [      0.       0.  108000.  108000. -108000.  216000.]]
d =
[0.0000e+00 0.0000e+00 0.0000e+00 3.9097e-05 0.0000e+00 0.0000e+00]
reaction =
[38.0125 21.1125 33.79    0.     -4.2225  4.2225]
